In [4]:
import numpy as np
import time

def simulate_full_control(N=4, runs=100):
    # Simulate periods, execution times, and setpoints
    periods = np.random.uniform(100, 200, N)  # Initial periods in ms
    exec_times = np.random.uniform(10, 80, N) # Simulated observed exec times
    setpoints = np.random.uniform(0.2, 0.8, N)

    # Simulate gain matrix (diagonal dominant + coupling)
    base_diag = 0.89 * 10
    off_diag = 0.39 / 10
    A = np.full((N, N), off_diag)
    np.fill_diagonal(A, base_diag)

    def control_logic(periods, exec_times, setpoints, A):
        utilizations = exec_times / periods
        slack = setpoints - utilizations
        delta = A @ slack
        total_period_sum = np.sum(periods)
        new_periods = np.zeros(N)
        for i in range(N):
            denom = 1 + periods[i] * (total_period_sum - periods[i]) * delta[i]
            new_periods[i] = periods[i] / denom if denom != 0 else periods[i]
        return new_periods

    # Warm-up
    control_logic(periods, exec_times, setpoints, A)

    # Timed run
    start = time.time()
    for _ in range(runs):
        _ = control_logic(periods, exec_times, setpoints, A)
    end = time.time()

    avg_overhead_us = ((end - start) / runs) * 1_000_000  # microseconds
    return avg_overhead_us

results = {}
for n_tasks in [4, 8, 16]:
    overhead = simulate_full_control(N=n_tasks)
    results[n_tasks] = overhead

results

{4: 14.352798461914062, 8: 17.969608306884766, 16: 25.391578674316406}